In [ ]:
# pip install tensorflow

tensorflow

In [3]:
import tensorflow as tf

In [ ]:
# 1. 필요한 라이브러리 임포트
# TensorFlow와 Keras의 기능을사용하기위해 필요한모듈을불러옵니다.
import numpy as np
from tensorflow.keras.models import Sequential              # 순차적모델을생성하기위한 모듈
from tensorflow.keras.layers import Dense, Input, Dropout   # 밀집층(fully connected layer)을 추가하기위한 모듈
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split        # 데이터를학습/테스트세트로나누기위한 모듈
from sklearn.datasets import make_classification            # 예제 데이터셋생성 모듈

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [ ]:
# 2. 데이터 생성 및 전처리
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_classes=2,
    random_state=42
)

In [ ]:
# 데이터 분리 : 80 : 20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [ ]:
# 3. 모델 생성
model = Sequential([
    # Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  # 첫번째 레이어층, 입력 데이터 사이즈들 정의
    Input(shape=(X_train.shape[1],)),
    Dense(8, activation='relu'),    # 은닉층 : 핵심 레이어 => 학습의 복잡성에 따라 여러개 배치 가능
    Dense(1, activation='sigmoid')
])

In [ ]:
# 4. 모델 컴파일
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [11]:
# 5. 모델 훈련
history=model.fit(
    X_train, y_train,           # 학습용 데이터와 레이블
    validation_split=0.2,       # 검증 데이터 비율 (학습 데이터의 20%)
    epochs=20,                  # 학습 반복 횟수
    batch_size=32,              # 한 번의 학습에서 사용하는 데이터 샘플 수
    verbose=1                   # 학습 진행 상태를 출력
)

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7625 - loss: 0.5136 - val_accuracy: 0.7563 - val_loss: 0.5167
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7719 - loss: 0.4993 - val_accuracy: 0.7688 - val_loss: 0.5009
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7891 - loss: 0.4860 - val_accuracy: 0.7750 - val_loss: 0.4869
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8016 - loss: 0.4741 - val_accuracy: 0.7937 - val_loss: 0.4733
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8047 - loss: 0.4627 - val_accuracy: 0.8313 - val_loss: 0.4609
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8047 - loss: 0.4525 - val_accuracy: 0.8313 - val_loss: 0.4488
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8094 - loss: 0.4429 - val_accuracy: 0.8313 - val_loss: 0.4375
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8188 - loss: 0.4339 - val_accuracy: 0.8375 - val_loss:

In [ ]:
# 6. 모델 평가
test_loss, test_acc = model.evaluate(X_test, y_test)
print(test_loss)
print(test_acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8400 - loss: 0.4156  
0.41556090116500854
0.8399999737739563


In [14]:
# 7. 모델 예측
prediction = model.predict(X_test[:5])  # 테스트 데이터 중 5개의 샘플 예측
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
[[0.68987906]
 [0.47370368]
 [0.36635128]
 [0.881907  ]
 [0.9188396 ]]


In [15]:
# 과적합 방지
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_classes=2,
    random_state=42
)

In [16]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
    # 7:3
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
    # 30 -> 15 : 15
)

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1], )),  # 입력층
    Dense(64, activation='relu'),   # 첫번째 은닉층
    
    # 과적합 방지
    Dropout(0.5),

    Dense(32, activation='relu'),   # 두번째 은닉층
    Dense(1, activation='sigmoid')  # 출력층 : 이진분류
])

In [19]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [20]:
# 과적합 방지
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [22]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping]
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9114 - loss: 0.2304 - val_accuracy: 0.8667 - val_loss: 0.3127
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9086 - loss: 0.2320 - val_accuracy: 0.8733 - val_loss: 0.3099
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9171 - loss: 0.2250 - val_accuracy: 0.8733 - val_loss: 0.3118
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9114 - loss: 0.2143 - val_accuracy: 0.8733 - val_loss: 0.3095
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9229 - loss: 0.2087 - val_accuracy: 0.8667 - val_loss: 0.3145
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9214 - loss: 0.2073 - val_accuracy: 0.8667 - val_loss: 0.3180
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9257 - loss: 0.2266 - val_accuracy: 0.8733 - val_loss: 0.3142
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9200 - loss: 0.2174 - val_accuracy: 0.8733 - v

In [23]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(test_loss)
print(test_acc)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8267 - loss: 0.4796 
0.4796123802661896
0.8266666531562805


다중분류

In [24]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist

In [25]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [26]:
# 이미지 데이터 => 1차원 벡터
X_train = X_train.reshape(-1, 784)/255.0 # 28*28 => 784
X_test = X_test.reshape(-1, 784)/255.0

In [27]:
# 원-핫인코딩
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

In [28]:
model = Sequential([
    Input(shape=(784, )),
    Dense(64, activation='relu'),

    # 출력층
    Dense(10, activation='softmax')
])

In [29]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [32]:
model.fit(
    X_train, y_train,
    epochs = 10,
    batch_size = 32,
    validation_split=0.2
)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9073 - loss: 0.3315 - val_accuracy: 0.9410 - val_loss: 0.2000
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9523 - loss: 0.1644 - val_accuracy: 0.9578 - val_loss: 0.1494
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9650 - loss: 0.1195 - val_accuracy: 0.9636 - val_loss: 0.1243
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9726 - loss: 0.0930 - val_accuracy: 0.9674 - val_loss: 0.1083
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9780 - loss: 0.0750 - val_accuracy: 0.9703 - val_loss: 0.0995
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9816 - loss: 0.0625 - val_accuracy: 0.9713 - val_loss: 0.0983
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9848 - loss: 0.0529 - val_accuracy: 0.9718 - val_loss: 0.0980
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9871 - loss: 0.0439 - 

In [33]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(test_loss)
print(test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9743 - loss: 0.0921
0.09208283573389053
0.9743000268936157


In [34]:
# 예측
prediction = model.predict(X_test[:1])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step


In [35]:
print(prediction)

[[4.1587147e-09 2.4018765e-09 4.5847830e-08 1.4456050e-04 6.4677409e-11
  2.4641735e-09 5.4356203e-12 9.9983954e-01 9.9114352e-08 1.5689719e-05]]


In [36]:
prediction[0]

array([4.1587147e-09, 2.4018765e-09, 4.5847830e-08, 1.4456050e-04,
       6.4677409e-11, 2.4641735e-09, 5.4356203e-12, 9.9983954e-01,
       9.9114352e-08, 1.5689719e-05], dtype=float32)

In [37]:
print(prediction.argmax())

7
